In [ ]:
#Check fp1fp2 contamination

In [3]:
"""
check_fp1fp2_contamination.py
==============================
Scans every subject folder under PREPROCESSED_ROOT, loads the cleaned
EEG .fif file, and checks whether Fp1 / Fp2 still carry residual
low-frequency (delta-band) contamination that was NOT removed by the
original ICA step.

How it works
------------
Fp1 / Fp2 contamination (mainly slow EOG drift and sweat artifact) shows
up as abnormally high power in the 1–4 Hz delta band relative to
neighbouring frontal channels (Fz, FC1, FC2, AF3, AF4, F3, F4).
A z-score is computed per subject:

    z_fp = (delta_power_fp12_mean - median_frontal_delta) / mad_frontal_delta

Subjects where z_fp > Z_THRESHOLD (default 2.5) are flagged.
A second check looks at peak-to-peak amplitude in a 1–4 Hz bandpass.

Output
------
- Console table with status per subject
- check_fp1fp2_report.csv  saved next to this script
- Optional: matplotlib figure with PSD comparison (set PLOT=True)

Usage
-----
    python check_fp1fp2_contamination.py

Edit the three constants at the top to match your system:
    PREPROCESSED_ROOT  – folder produced by Preprocessing_v2_run_a1b_10s.py
    SUBJECTS           – list of subject IDs, or None to auto-discover
    PLOT               – True / False
"""

import os
import csv
import warnings
import numpy as np

# ── user-editable ──────────────────────────────────────────────────────────────
PREPROCESSED_ROOT = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s"
SUBJECTS = None          # None  → auto-discover all VPxxx folders
Z_THRESHOLD  = 2.5       # flag if delta z-score > this
P2P_RATIO_THRESHOLD = 2.0  # flag if Fp1/Fp2 peak-to-peak ratio > this vs frontal median
DELTA_BAND   = (1.0, 4.0)  # Hz – slow-drift / EOG band
FRONTAL_REF  = ["Fz", "FC1", "FC2", "AF3", "AF4", "F3", "F4",
                "F5", "F6", "AFz", "AFF1h", "AFF2h"]  # neighbours used as reference
PLOT         = True       # set False if matplotlib is not available
# ──────────────────────────────────────────────────────────────────────────────

try:
    import mne
    mne.set_log_level("WARNING")
except ImportError:
    raise ImportError("MNE-Python is required: pip install mne")


# ── helpers ───────────────────────────────────────────────────────────────────

def _channel_delta_power(raw, ch_names_subset, fmin=1.0, fmax=4.0):
    """
    Return mean log10 power in [fmin, fmax] Hz for the given channels.
    Uses Welch PSD via MNE compute_psd().
    """
    picks = mne.pick_channels(raw.ch_names, include=ch_names_subset,
                               ordered=False)
    if len(picks) == 0:
        return None
    # NEW — n_fft=512 works even for very short recordings (2.56s at 200Hz):
    n_samples = raw.get_data(picks=picks).shape[1]
    n_fft = min(512, n_samples)
    psd = raw.compute_psd(method="welch", fmin=fmin, fmax=fmax,
                          picks=picks, n_fft=n_fft, verbose=False)
    power = psd.get_data()          # (n_ch, n_freqs)
    mean_power = power.mean(axis=1) # average over frequency bins
    return np.log10(mean_power + 1e-30)   # log scale, shape (n_ch,)


def _channel_ptp_delta(raw, ch_names_subset, fmin=1.0, fmax=4.0):
    """
    Peak-to-peak amplitude of the delta-filtered signal (µV) per channel.
    """
    picks = mne.pick_channels(raw.ch_names, include=ch_names_subset,
                               ordered=False)
    if len(picks) == 0:
        return None
    raw_delta = raw.copy().filter(fmin, fmax, picks=picks,
                                   method="fir", verbose=False)
    data = raw_delta.get_data(picks=picks) * 1e6   # V → µV
    return np.ptp(data, axis=1)    # shape (n_ch,)


def check_subject(subject_id, fif_path):
    """
    Load the cleaned .fif, run both checks, return a result dict.
    """
    result = {
        "subject":         subject_id,
        "fif_found":       os.path.exists(fif_path),
        "fp1_present":     False,
        "fp2_present":     False,
        "n_ref_channels":  0,
        "delta_z_fp1":     None,
        "delta_z_fp2":     None,
        "delta_z_mean":    None,
        "ptp_ratio_fp1":   None,
        "ptp_ratio_fp2":   None,
        "flagged_z":       False,
        "flagged_ptp":     False,
        "flagged":         False,
        "note":            "",
    }

    if not result["fif_found"]:
        result["note"] = "fif not found"
        return result

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)

    ch_upper = [c.upper() for c in raw.ch_names]
    result["fp1_present"] = "FP1" in ch_upper
    result["fp2_present"] = "FP2" in ch_upper

    fp_channels = []
    if result["fp1_present"]:
        fp_channels.append(raw.ch_names[ch_upper.index("FP1")])
    if result["fp2_present"]:
        fp_channels.append(raw.ch_names[ch_upper.index("FP2")])

    if not fp_channels:
        result["note"] = "Fp1 and Fp2 not present (already dropped/interpolated)"
        return result

    # reference channels available in this recording
    ref_available = [c for c in raw.ch_names
                     if c in FRONTAL_REF and c not in fp_channels]
    result["n_ref_channels"] = len(ref_available)

    if len(ref_available) < 2:
        result["note"] = "too few reference channels for comparison"
        return result

    # ── Check 1: delta-band log-power z-score ──────────────────────────────
    ref_log_power = _channel_delta_power(raw, ref_available,
                                          fmin=DELTA_BAND[0], fmax=DELTA_BAND[1])
    fp_log_power  = _channel_delta_power(raw, fp_channels,
                                          fmin=DELTA_BAND[0], fmax=DELTA_BAND[1])

    if ref_log_power is not None and fp_log_power is not None:
        ref_median = np.median(ref_log_power)
        ref_mad    = np.median(np.abs(ref_log_power - ref_median))
        ref_mad    = max(ref_mad, 1e-6)      # avoid division by zero

        z_scores = (fp_log_power - ref_median) / ref_mad

        fp1_idx = 0
        fp2_idx = 1 if len(fp_channels) > 1 else None

        result["delta_z_fp1"] = float(z_scores[fp1_idx]) if "FP1" in [c.upper() for c in fp_channels] else None
        result["delta_z_fp2"] = float(z_scores[fp2_idx]) if fp2_idx is not None else None
        result["delta_z_mean"] = float(np.mean(z_scores))

        if any(z > Z_THRESHOLD for z in z_scores):
            result["flagged_z"] = True

    # ── Check 2: peak-to-peak amplitude ratio ──────────────────────────────
    ref_ptp = _channel_ptp_delta(raw, ref_available,
                                  fmin=DELTA_BAND[0], fmax=DELTA_BAND[1])
    fp_ptp  = _channel_ptp_delta(raw, fp_channels,
                                  fmin=DELTA_BAND[0], fmax=DELTA_BAND[1])

    if ref_ptp is not None and fp_ptp is not None:
        ref_median_ptp = np.median(ref_ptp)
        if ref_median_ptp > 0:
            ratios = fp_ptp / ref_median_ptp

            if "FP1" in [c.upper() for c in fp_channels]:
                result["ptp_ratio_fp1"] = float(ratios[0])
            if len(fp_channels) > 1:
                result["ptp_ratio_fp2"] = float(ratios[1])

            if any(r > P2P_RATIO_THRESHOLD for r in ratios):
                result["flagged_ptp"] = True

    # overall flag = either check positive
    result["flagged"] = result["flagged_z"] or result["flagged_ptp"]

    if result["flagged"]:
        reasons = []
        if result["flagged_z"]:
            reasons.append(f"delta_z={result['delta_z_mean']:.2f}")
        if result["flagged_ptp"]:
            fp1r = result["ptp_ratio_fp1"]
            fp2r = result["ptp_ratio_fp2"]
            reasons.append(f"ptp_ratio_fp1={fp1r:.2f}" if fp1r else "")
            reasons.append(f"ptp_ratio_fp2={fp2r:.2f}" if fp2r else "")
        result["note"] = " | ".join(r for r in reasons if r)

    return result, raw


# ── main ─────────────────────────────────────────────────────────────────────

def main():
    # discover subjects
    if SUBJECTS is not None:
        subject_ids = SUBJECTS
    else:
        subject_ids = sorted([
            d for d in os.listdir(PREPROCESSED_ROOT)
            if os.path.isdir(os.path.join(PREPROCESSED_ROOT, d))
            and d.startswith("VP")
        ])

    print(f"\nScanning {len(subject_ids)} subjects in:\n  {PREPROCESSED_ROOT}\n")
    print(f"{'Subject':<10} {'Fp1':>6} {'Fp2':>6} {'z_Fp1':>7} {'z_Fp2':>7} "
          f"{'ptp_Fp1':>8} {'ptp_Fp2':>8} {'Status'}")
    print("-" * 75)

    results = []
    raws_for_plot = {}

    for sid in subject_ids:
        fif_path = os.path.join(PREPROCESSED_ROOT, sid,
                                 f"{sid}_eeg_clean_raw.fif")
        out = check_subject(sid, fif_path)

        # check_subject returns (dict, raw) if fif was loaded, else just dict
        if isinstance(out, tuple):
            res, raw = out
            if PLOT and res["flagged"]:
                raws_for_plot[sid] = raw
        else:
            res = out

        results.append(res)

        status = "⚠  FLAGGED" if res["flagged"] else ("OK" if res["fif_found"] else "MISSING")
        fp1_str  = ("yes" if res["fp1_present"] else "—")
        fp2_str  = ("yes" if res["fp2_present"] else "—")
        z1  = f"{res['delta_z_fp1']:.2f}"  if res["delta_z_fp1"]  is not None else "—"
        z2  = f"{res['delta_z_fp2']:.2f}"  if res["delta_z_fp2"]  is not None else "—"
        r1  = f"{res['ptp_ratio_fp1']:.2f}" if res["ptp_ratio_fp1"] is not None else "—"
        r2  = f"{res['ptp_ratio_fp2']:.2f}" if res["ptp_ratio_fp2"] is not None else "—"

        print(f"{sid:<10} {fp1_str:>6} {fp2_str:>6} {z1:>7} {z2:>7} "
              f"{r1:>8} {r2:>8}   {status}")
        if res["note"]:
            print(f"{'':10}  → {res['note']}")

    # summary
    flagged  = [r["subject"] for r in results if r["flagged"]]
    ok       = [r["subject"] for r in results if not r["flagged"] and r["fif_found"]]
    missing  = [r["subject"] for r in results if not r["fif_found"]]

    print("\n" + "=" * 75)
    print(f"  FLAGGED  ({len(flagged)}): {', '.join(flagged) if flagged else 'none'}")
    print(f"  CLEAN    ({len(ok)}): {', '.join(ok)}")
    if missing:
        print(f"  MISSING  ({len(missing)}): {', '.join(missing)}")
    print("=" * 75)

    if flagged:
        print("\nRECOMMENDED ACTION for flagged subjects:")
        print("  Run ICA-again_RunA1b.py with FORCED_BAD_CHANNELS = {")
        for s in flagged:
            r = next(x for x in results if x["subject"] == s)
            forced = []
            if r["fp1_present"] and (r["delta_z_fp1"] or 0) > Z_THRESHOLD:
                forced.append("Fp1")
            if r["fp2_present"] and (r["delta_z_fp2"] or 0) > Z_THRESHOLD:
                forced.append("Fp2")
            if not forced:
                forced = ["Fp1", "Fp2"]  # fallback: force both
            print(f'    "{s}": {forced},')
        print("  }")

    # save CSV report
    csv_path = os.path.join(os.path.dirname(__file__), "check_fp1fp2_report.csv")
    fieldnames = ["subject", "fif_found", "fp1_present", "fp2_present",
                  "n_ref_channels", "delta_z_fp1", "delta_z_fp2", "delta_z_mean",
                  "ptp_ratio_fp1", "ptp_ratio_fp2",
                  "flagged_z", "flagged_ptp", "flagged", "note"]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"\nCSV report saved to: {csv_path}")

    # optional plot
    if PLOT and raws_for_plot:
        _make_psd_plots(raws_for_plot)


def _make_psd_plots(raws_dict):
    """PSD overlay: Fp1/Fp2 vs frontal reference channels, one panel per subject."""
    try:
        import matplotlib
        matplotlib.use("Agg")   # non-interactive backend — remove if you want popup
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not available – skipping plots")
        return

    n = len(raws_dict)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)

    for ax, (sid, raw) in zip(axes[0], raws_dict.items()):
        ch_upper = [c.upper() for c in raw.ch_names]

        fp_chs  = [raw.ch_names[ch_upper.index(c)]
                   for c in ("FP1", "FP2") if c in ch_upper]
        ref_chs = [c for c in raw.ch_names if c in FRONTAL_REF]

        for ch in ref_chs:
            psd = raw.compute_psd(method="welch", fmin=0.5, fmax=20.0,
                                   picks=ch, verbose=False)
            freqs = psd.freqs
            power = psd.get_data()[0]
            ax.semilogy(freqs, power, color="steelblue", alpha=0.4, linewidth=0.8)

        for ch in fp_chs:
            psd = raw.compute_psd(method="welch", fmin=0.5, fmax=20.0,
                                   picks=ch, verbose=False)
            freqs = psd.freqs
            power = psd.get_data()[0]
            ax.semilogy(freqs, power, linewidth=2.0,
                        label=ch, color="crimson" if "1" in ch else "orangered")

        ax.axvspan(*DELTA_BAND, alpha=0.12, color="red", label="delta band")
        ax.set_title(f"{sid} — PSD (flagged)", fontsize=11)
        ax.set_xlabel("Frequency (Hz)")
        ax.set_ylabel("Power (V²/Hz)")
        ax.legend(fontsize=8)
        ax.grid(True, which="both", alpha=0.3)

    fig.suptitle("Fp1/Fp2 PSD vs frontal reference channels\n"
                 "(red=Fp1/Fp2, blue=reference, shaded=delta band)",
                 fontsize=12)
    plt.tight_layout()
    out_path = os.path.join(os.path.dirname(__file__), "fp1fp2_psd_flagged.png")
    plt.savefig(out_path, dpi=120)
    print(f"PSD plot saved to: {out_path}")
    plt.close(fig)


if __name__ == "__main__":
    main()



Scanning 26 subjects in:
  E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s

Subject       Fp1    Fp2   z_Fp1   z_Fp2  ptp_Fp1  ptp_Fp2 Status
---------------------------------------------------------------------------
VP001         yes    yes    1.66    1.89     1.56     2.44   ⚠  FLAGGED
            → ptp_ratio_fp1=1.56 | ptp_ratio_fp2=2.44
VP002         yes    yes   -3.54    2.67     0.74     0.80   ⚠  FLAGGED
            → delta_z=-0.44
VP003         yes    yes    2.55    1.13     1.82     1.25   ⚠  FLAGGED
            → delta_z=1.84


C:\Users\SPACE\myenv\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 512 is greater than input length  = 300, using nperseg = 300
  return _func(*args, **kwargs)
C:\Users\SPACE\myenv\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 512 is greater than input length  = 300, using nperseg = 300
  return _func(*args, **kwargs)
C:\Users\SPACE\myenv\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 512 is greater than input length  = 100, using nperseg = 100
  return _func(*args, **kwargs)
C:\Users\SPACE\myenv\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 512 is greater than input length  = 300, using nperseg = 300
  return _func(*args, **kwargs)
C:\Users\SPACE\myenv\Lib\site-packages\mne\time_frequency\psd.py:257: UserWarning: nperseg = 512 is greater than input length  = 300, using nperseg = 300
  return _func(*args, **kwargs)
C:\Users\SPACE\myenv\Lib\site-packages\mne\time_frequency\psd.py:257: 

VP004         yes    yes   -2.20   18.71     1.58     6.51   ⚠  FLAGGED
            → delta_z=8.26 | ptp_ratio_fp1=1.58 | ptp_ratio_fp2=6.51
VP005         yes    yes   12.81   14.19     3.97     4.39   ⚠  FLAGGED
            → delta_z=13.50 | ptp_ratio_fp1=3.97 | ptp_ratio_fp2=4.39
VP006         yes    yes   22.18   21.37     4.41     3.61   ⚠  FLAGGED
            → delta_z=21.78 | ptp_ratio_fp1=4.41 | ptp_ratio_fp2=3.61
VP007         yes    yes    5.87    4.27     1.86     1.62   ⚠  FLAGGED
            → delta_z=5.07
VP008         yes    yes   15.44   17.20     1.45     1.35   ⚠  FLAGGED
            → delta_z=16.32
VP009         yes    yes   12.17   11.46     0.84     0.77   ⚠  FLAGGED
            → delta_z=11.81
VP010         yes    yes   -0.79   -2.09     2.30     0.76   ⚠  FLAGGED
            → ptp_ratio_fp1=2.30 | ptp_ratio_fp2=0.76
VP011         yes    yes  -13.05  -11.22     0.51     0.51   OK
VP012         yes    yes   36.07   55.42     1.12     2.06   ⚠  FLAGGED
            → 

NameError: name '__file__' is not defined